# OralVerse — MeshSegNet Training (Kaggle)

**Platform**: Kaggle Notebooks — free P100 GPU, 30 GPU hrs/week

**Setup**:
1. New Notebook → Settings (right panel) → Accelerator: **GPU P100**
2. Settings → Internet: **On** (needed to clone repo and download dataset)
3. Fill in your GitHub repo URL in cell 2
4. Run All (`Run` menu → `Run All`)
5. Come back in ~5 hours
6. Download `.pt` files from the **Output** tab on the right

**Expected output**: `meshsegnet_upper_best.pt` + `meshsegnet_lower_best.pt`  
Place them in `ai/orthodontics/segmentation/meshsegnet/checkpoints/` locally.

## 1. Check GPU

In [ ]:
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU     : {gpu.name}")
    print(f"VRAM    : {gpu.total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError(
        "No GPU detected.\n"
        "Go to: Settings (right panel) → Accelerator → GPU P100"
    )

## 2. Clone repo

In [ ]:
import os
from pathlib import Path

REPO_URL  = "https://github.com/sh1v3n/OralVerse-CAD.git"  # ← your repo
BRANCH    = "claude/features"
WORK_DIR  = Path("/kaggle/working/OralVerse-CAD")

if not WORK_DIR.exists():
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {WORK_DIR}
else:
    !cd {WORK_DIR} && git pull

os.chdir(WORK_DIR)
print(f"Working directory: {Path.cwd()}")

## 3. Install dependencies

In [ ]:
REQ = WORK_DIR / "ai/orthodontics/segmentation/meshsegnet/requirements.txt"
if REQ.exists():
    !pip install -q -r {REQ}
else:
    print(f"requirements.txt not found at {REQ} — installing directly")
    !pip install -q trimesh tqdm pyyaml numpy
print("Dependencies installed.")

## 4. Download dataset (Teeth3DS+ on OSF)

`N_PARTS = 1` fetches only `data_part_1.zip` (~1.5 GB, ~300 scans) — enough for a good model in ~5 hrs.  
Set `N_PARTS = 7` for the full 2100-scan dataset (~10.5 GB, ~4× longer).

In [ ]:
import json
import urllib.request
import zipfile

# Full dataset: 7 parts = 2100 scans (~10.5 GB, ~10 hrs training)
N_PARTS = 7

RAW_DIR  = WORK_DIR / "ai/orthodontics/segmentation/meshsegnet/raw_data"
DATA_DIR = WORK_DIR / "ai/orthodontics/segmentation/meshsegnet/data"
CKPT_DIR = WORK_DIR / "ai/orthodontics/segmentation/meshsegnet/checkpoints"

for d in [RAW_DIR, DATA_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OSF_PARTS = {
    "data_part_1": "5cmg3",
    "data_part_2": "xfcn9",
    "data_part_3": "hw7bj",
    "data_part_4": "n9bd7",
    "data_part_5": "7az58",
    "data_part_6": "2ybr4",
    "data_part_7": "gvjah",
}


def _get_osf_zip_url(node_id):
    api = f"https://api.osf.io/v2/nodes/{node_id}/files/osfstorage/?format=json"
    with urllib.request.urlopen(api) as r:
        data = json.load(r)
    for item in data["data"]:
        if item["attributes"]["name"].endswith(".zip"):
            return item["links"]["download"], item["attributes"]["name"]
    raise RuntimeError(f"No .zip found in OSF node {node_id}")


part_names = list(OSF_PARTS.keys())[:N_PARTS]

for part_name in part_names:
    node_id = OSF_PARTS[part_name]
    dest = RAW_DIR / f"{part_name}.zip"

    # Skip if already extracted (zip was deleted after extraction)
    marker = RAW_DIR / f".done_{part_name}"
    if marker.exists():
        print(f"[skip] {part_name} already extracted")
        continue

    print(f"Resolving download URL for {part_name} ...")
    url, fname = _get_osf_zip_url(node_id)
    print(f"Downloading {fname} (~1.5 GB) ...")
    urllib.request.urlretrieve(url, dest)

    print(f"Extracting {part_name}.zip ...")
    with zipfile.ZipFile(dest, "r") as zf:
        zf.extractall(RAW_DIR)

    dest.unlink()       # free ~1.5 GB before next download
    marker.touch()      # mark as done so re-runs can skip
    print(f"Done — zip deleted to free space.")

n_obj = len(list(RAW_DIR.rglob("*.obj")))
print(f"\nDataset ready: {n_obj} OBJ files")
if n_obj == 0:
    raise RuntimeError("No OBJ files found — check download above")

## 5. Preprocess scans → .npz

In [ ]:
!python -m ai.orthodontics.segmentation.meshsegnet.preprocess \
    --data_dir {RAW_DIR} \
    --out_dir  {DATA_DIR} \
    --workers  4

n_npz = len(list(DATA_DIR.rglob("*.npz")))
print(f"\nPreprocessed: {n_npz} .npz files")

## 6. Train — Upper arch

In [ ]:
import subprocess, sys

EPOCHS    = 100    # full run on 2100 scans
MAX_FACES = 12000  # reduce to 8000 if you hit OOM

subprocess.run([
    sys.executable, "-m", "ai.orthodontics.segmentation.meshsegnet.train",
    "--data_dir", str(DATA_DIR),
    "--arch",     "upper",
    "--out_dir",  str(CKPT_DIR),
    "--epochs",   str(EPOCHS),
    "--max_faces", str(MAX_FACES),
], check=True)

## 7. Train — Lower arch

In [ ]:
subprocess.run([
    sys.executable, "-m", "ai.orthodontics.segmentation.meshsegnet.train",
    "--data_dir", str(DATA_DIR),
    "--arch",     "lower",
    "--out_dir",  str(CKPT_DIR),
    "--epochs",   str(EPOCHS),
    "--max_faces", str(MAX_FACES),
], check=True)

## 8. Evaluate

In [ ]:
for arch in ["upper", "lower"]:
    ckpt = CKPT_DIR / f"meshsegnet_{arch}_best.pt"
    if ckpt.exists():
        subprocess.run([
            sys.executable, "-m", "ai.orthodontics.segmentation.meshsegnet.evaluate",
            "--checkpoint", str(ckpt),
            "--data_dir",   str(DATA_DIR),
            "--split",      "test",
        ], check=True)
    else:
        print(f"No checkpoint found for {arch} — check training output above.")

## 9. Summary

Your checkpoints are now in the **Output** tab on the right side of Kaggle.  
Download them and place locally at:
```
ai/orthodontics/segmentation/meshsegnet/checkpoints/
  meshsegnet_upper_best.pt
  meshsegnet_lower_best.pt
```

Then verify locally:
```bash
python -m ai.orthodontics.segmentation.meshsegnet.export_model \
    --checkpoint ai/orthodontics/segmentation/meshsegnet/checkpoints/meshsegnet_upper_best.pt
```

Then activate:
```bash
export ORALVERSE_SEGMENTER=meshsegnet
export MESHSEGNET_WEIGHTS=/path/to/checkpoints/meshsegnet_upper_best.pt
```

In [ ]:
checkpoints = list(CKPT_DIR.glob("*.pt"))
if not checkpoints:
    print("No checkpoints found — check training logs above for errors.")
else:
    print(f"Checkpoints ({len(checkpoints)}):")
    for c in sorted(checkpoints):
        print(f"  {c.name}  ({c.stat().st_size / 1e6:.1f} MB)")
    print("\nDownload from the Output tab on the right.")